# 🔍 Fake News Detection using LSTM

**Goal:** Automatically classify news articles as REAL or FAKE using a stacked LSTM neural network.

**Tech Stack:** Python · TensorFlow · Keras · NLTK · NumPy · Pandas · Matplotlib

---
### Pipeline Overview
```
Raw Text → Preprocessing → Tokenization → Padding → Embedding → LSTM → Classification
```

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

# Deep Learning
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping

# Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded successfully!')

## 2. Load Dataset

In [ ]:
# Load dataset
# Dataset contains two columns: 'text' (news article) and 'label' (0=REAL, 1=FAKE)
df = pd.read_csv('news_dataset.csv')

print('Shape:', df.shape)
print('\nLabel distribution:')
print(df['label'].value_counts())
print('\nSample rows:')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['label'].value_counts()
axes[0].bar(['REAL', 'FAKE'], counts.values, color=['#0F7B55', '#E03E3E'], alpha=0.85)
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Article length distribution
df['text_length'] = df['text'].apply(lambda x: len(str(x).split()))
axes[1].hist(df[df['label']==0]['text_length'], bins=50, alpha=0.7, color='#0F7B55', label='REAL')
axes[1].hist(df[df['label']==1]['text_length'], bins=50, alpha=0.7, color='#E03E3E', label='FAKE')
axes[1].set_title('Article Length Distribution', fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Text Preprocessing Pipeline

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Full text preprocessing pipeline:
    1. Lowercase
    2. Remove punctuation & special characters
    3. Remove stopwords
    4. Return cleaned text
    """
    # Lowercase
    text = str(text).lower()
    # Remove punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize and remove stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

# Apply preprocessing
df['cleaned_text'] = df['text'].apply(preprocess_text)

print('Sample original text:')
print(df['text'].iloc[0][:200])
print('\nSample cleaned text:')
print(df['cleaned_text'].iloc[0][:200])

## 5. Tokenization & Sequence Padding

In [ ]:
# Hyperparameters
MAX_VOCAB_SIZE = 20000   # Keep top 20,000 most frequent words
MAX_SEQ_LENGTH = 500     # Pad/truncate all sequences to 500 tokens
EMBEDDING_DIM  = 128     # Word vector dimensions

# Tokenize
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(df['cleaned_text'])

# Convert text to integer sequences
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])

# Pad sequences to equal length
X = pad_sequences(sequences, maxlen=MAX_SEQ_LENGTH, padding='post', truncating='post')
y = df['label'].values

print(f'Vocabulary size : {len(tokenizer.word_index):,}')
print(f'Input shape     : {X.shape}')
print(f'Sample sequence : {X[0][:15]}...')

## 6. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Testing samples  : {X_test.shape[0]:,}')
print(f'Train label dist : REAL={sum(y_train==0):,} | FAKE={sum(y_train==1):,}')

## 7. Build LSTM Model Architecture

In [ ]:
model = Sequential([
    # Word Embedding Layer
    Embedding(input_dim=MAX_VOCAB_SIZE,
              output_dim=EMBEDDING_DIM,
              input_length=MAX_SEQ_LENGTH),

    # Spatial Dropout to reduce overfitting on embeddings
    SpatialDropout1D(0.2),

    # LSTM Layer 1 — return sequences for stacking
    LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True),

    # LSTM Layer 2 — final LSTM layer
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),

    # Dropout for regularization
    Dropout(0.5),

    # Output Layer — sigmoid for binary classification
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 8. Train the Model

In [ ]:
# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='#0F7B55', lw=2.5, marker='o', ms=5)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='#C97A00', lw=2.5, marker='s', ms=5)
axes[0].set_title('Model Accuracy per Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.4)

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', color='#E03E3E', lw=2.5, marker='o', ms=5)
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='#C2185B', lw=2.5, marker='s', ms=5)
axes[1].set_title('Model Loss per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('images/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved!')

## 10. Model Evaluation

In [ ]:
# Predictions
y_pred_prob = model.predict(X_test)
y_pred      = (y_pred_prob > 0.5).astype(int).flatten()

# Metrics
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['REAL', 'FAKE']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['REAL', 'FAKE'],
            yticklabels=['REAL', 'FAKE'],
            linewidths=1)
plt.title('Confusion Matrix — Fake News Detection', fontweight='bold', fontsize=13)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.savefig('images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved!')

## 11. Save the Trained Model

In [ ]:
# Save model in HDF5 format
model.save('fake_news_model.h5')
print('Model saved as: fake_news_model.h5')

# Save tokenizer for later use
import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print('Tokenizer saved as: tokenizer.pkl')

## 12. Predict on New Text
Use the trained model to classify any news article.

In [ ]:
def predict_news(text, model, tokenizer, max_len=500):
    """
    Predict if a news article is REAL or FAKE.
    Returns: label and confidence score
    """
    cleaned  = preprocess_text(text)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded   = pad_sequences(sequence, maxlen=max_len, padding='post')
    prob     = model.predict(padded, verbose=0)[0][0]
    label    = 'FAKE' if prob > 0.5 else 'REAL'
    confidence = prob if prob > 0.5 else 1 - prob
    return label, confidence

# Test on a sample
sample = "Scientists discover new vaccine that eliminates all forms of cancer in clinical trials"
label, conf = predict_news(sample, model, tokenizer)
print(f'Article : {sample}')
print(f'Prediction : {label}  (confidence: {conf*100:.1f}%)')